[실습 문제] 나만의 AI 해결사: 내 주변의 작은 문제 해결 앱 만들기 #38
1. 과제 개요 (Overview)
목표: 학교생활이나 일상생활에서 겪는 사소한 불편함이나 비효율적인 문제를 발견하고, LLM과 프롬프트 엔지니어링 기술을 활용하여 이를 해결하는 웹 애플리케이션(프로토타입)을 제작한다.

핵심 역량: 문제 해결 능력, 프롬프트 엔지니어링(Prompt Engineering), 기초 AI 애플리케이션 구현 능력.

2. 단계별 수행 가이드 (Step-by-Step Guide)

Step 1: 문제 발굴 및 정의 (Problem Definition)

나의 일상이나 친구들이 겪는 불편함을 관찰한다. 거창한 문제가 아니어도 된다.
예시:
"오늘 급식 메뉴를 보고 맛있는 조합인지 아닌지 판단하기 귀찮다."
"동아리 공지사항을 쓸 때마다 말투를 다듬는 게 어렵다."
"복잡한 가정통신문 내용을 3줄로 요약해서 보고 싶다."
과제: 해결하고자 하는 문제를 한 문장으로 정의하고, 타겟 사용자(나 자신 혹은 친구들)를 설정한다.

Step 2: 프롬프트 엔지니어링 (Prompt Engineering)

LLM이 문제를 해결하도록 지시하는 최적의 프롬프트를 설계한다.
필수 포함 요소:

Role (페르소나): AI에게 역할 부여 (예: "너는 영양사이자 메뉴 추천 전문가야.")
Context (맥락): 상황 설명 (예: "고등학생들이 좋아하는 입맛을 기준으로 분석해야 해.")

Task (지시 사항): 구체적인 수행 작업 (예: "메뉴 리스트를 입력받으면, 10점 만점으로 점수를 매기고 그 이유를 설명해 줘.")

Constraints (제약 조건): 출력 형식 제한 (예: "반말을 사용하고, 이모지를 3개 이상 포함해.")
과제: 다양한 프롬프트를 테스트해 보고, 가장 결과가 좋은 최종 프롬프트를 선정한다.

Step 3: 앱 구현 (Implementation)

선정된 프롬프트를 실제 작동하는 앱 형태로 만든다.
도구 추천:
GPT Chat Playground
Python + Streamlit (간단한 파이썬 코드로 웹 UI 제작)
과제: 사용자가 입력창에 텍스트를 넣으면, LLM이 처리하여 결과를 보여주는 인터페이스를 구현한다.

Step 4: 테스트 및 개선 (Testing & Iteration)

친구들에게 앱을 써보게 하고 피드백을 받는다.
LLM이 엉뚱한 대답을 하는 경우(할루시네이션 등), 프롬프트를 수정하여 성능을 개선한다.

3. 예시 시나리오 (Example Scenarios)
구분	앱 이름	기능 설명	프롬프트 핵심 전략
생활	냉장고 파먹기 AI	냉장고에 남은 재료만 입력하면 레시피 추천	"가진 재료만 써야 함", "자취생이 할 수 있는 쉬운 요리" 제약 조건 추가
학업	발표 대본 생성기	조사한 자료를 넣으면 발표용 대본으로 변환	"3분 분량으로 맞춰줘", "청중의 흥미를 끌 수 있는 유머 포함" 지시
소통	'너 T야?' 탐지기	카톡 메시지를 입력하면 공감 능력이 부족한지 판별	"공감형 대화 전문가" 페르소나 부여, 감정 분석 및 수정 제안

In [6]:
# Colab과 로컬 환경 모두 지원하는 API 키 설정
import os

try:
    # Colab 환경인 경우
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    print("✅ Colab 환경에서 API 키를 가져왔습니다.")
except ImportError:
    # 로컬 환경인 경우
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    if not OPENAI_API_KEY:
        # 환경변수가 없으면 직접 입력받기
        OPENAI_API_KEY = input("OPENAI_API_KEY를 입력하세요: ").strip()
        os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    print("✅ 로컬 환경에서 API 키를 설정했습니다.")

✅ Colab 환경에서 API 키를 가져왔습니다.


In [7]:
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
  model="gpt-4.1",
  messages=[
    {
      "role": "system",
      "content": [
        {
          "type": "text",
          "text": "# Role\n당신은 병원에서 근무하는 전문 간호사로서, 환자와 보호자가 이해하기 쉽도록 의사의 진료 내용을 친절하고 명확하게 설명하는 역할을 맡고 있습니다. 보호자는 의학 지식이 전혀 없다는 전제를 항상 기억하세요.\n\n# Context\n- 의료 설명을 할 때 절대 전문 용어나 어려운 단어를 사용하지 마세요. 반드시 쉬운 일상 언어로 풀어 설명해야 합니다.\n- 전달받은 진단, 검사 결과, 처방, 주의사항 등 이미 제공된 의사의 설명만을 다룹니다. 본인이 직접 진단하거나 추가적인 치료를 지시하지 않습니다.\n- 설명이 부족하거나 불확실한 정보가 있다면, 내용을 추측하거나 임의로 보충하지 말고 \"궁금하신 점은 병원에 다시 문의해 주세요\"와 같이 안내합니다.\n- 설명 내용은 5~10문장 내외의 자연스러운 한국어 대화체로 작성합니다.\n- 항상 차분하고 공감하는 태도와 어조를 유지하세요.\n\n# 출력 형식\n- [이해를 돕는 쉬운 설명] (5~10문장, 자연스럽고 공손한 대화체)\n- [불확실하거나 추가 확인 필요시 안내 문구] (필요할 경우에만)\n\n# 예시\n### 입력:  \n\"환자의 CT검사 결과, 좌측 대뇌반구에 소혈종이 관찰되었으며, 특별한 신경학적 증상은 없는 상태입니다.\"\n\n### 출력:  \n이해를 돕는 쉬운 설명:  \n\"검사 결과, 환자분 뇌의 왼쪽 부분에 아주 작은 피가 고인 자국이 발견되었습니다. 하지만 다행히 신경에 이상이 생기는 증상은 보이지 않고 있습니다. 앞으로도 혹시라도 불편한 증상이 생기면 바로 말씀해 주세요.\"  \n불확실하거나 추가 확인 필요시 안내 문구:  \n\"더 궁금한 점이 있으시면 언제든 병원에 문의해 주세요.\"\n\n### 입력:  \n\"혈액검사에서 백혈구 수치가 약간 높게 나왔으나, 현재 감염 증상은 관찰되지 않습니다.\"\n\n### 출력:  \n이해를 돕는 쉬운 설명:  \n\"피 검사에서 몸을 지키는 세포인 백혈구가 조금 높게 나왔지만, 지금은 감기나 다른 감염이 의심되는 증상은 보이지 않습니다. 앞으로 열이 나거나 상태가 달라지면 말씀해 주세요.\"  \n불확실하거나 추가 확인 필요시 안내 문구:  \n(필요 없음)\n\n(실제 입력 예시는 더 복잡하거나 길어지는 경우가 많으므로, 설명 부분은 상황에 따라 더 길어질 수 있습니다.)\n\n---\n\n**중요:**\n- 반드시 전문 용어를 풀어서 설명하세요.  \n- 절대 추가 진단이나 치료 제안, 임의의 해석을 하지 마세요.  \n- 필요한 경우 병원 문의를 안내하세요.\n\n(중요 목표: 보호자가 걱정 없이 진료 내용을 쉽게 이해할 수 있도록 돕는 것입니다.)"
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "다음은 의사가 보호자에게 설명한 진료 내용입니다.\n아래 형식에 맞춰 이해하기 쉽게 설명해 주세요.\n\n[출력 형식]\n1. 무슨 상태인지 (한 문장)\n2. 왜 문제가 되는지 (2~3문장)\n3. 당장 주의할 점 (불릿 3개)\n\n[진료 설명]\n이석이 반고리관으로 이동하면서 회전성 어지럼증이 발생하고 있으며,\n특정 자세 변화 시 증상이 악화됩니다.\n대부분은 자연 회복되지만 재발 가능성이 있습니다.\n"
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "이석증에 대해 설명해줘\n"
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "text",
          "text": "1. 무슨 상태인지  \n귀 안에 있는 아주 작은 돌조각이 제자리를 벗어나서, 머리를 움직일 때 어지럼증이 생기는 상태입니다.\n\n2. 왜 문제가 되는지  \n이 작은 돌조각이 움직이면 뇌가 몸이 움직였다고 잘못 느껴서, 세상이 빙글빙글 도는 것 같은 어지러움을 느끼게 됩니다. 특히 누웠다 일어나거나 고개를 돌릴 때 증상이 더 심해질 수 있습니다. 대부분은 시간이 지나면 저절로 좋아지지만, 다시 생길 수도 있습니다.\n\n3. 당장 주의할 점  \n- 갑자기 머리를 확 움직이지 않도록 조심하세요.  \n- 어지러울 때는 잠시 쉬고, 넘어지지 않게 주의하세요.  \n- 증상이 심해지거나 오래 계속되면 병원에 다시 오세요."
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "백내장에 대해 설명해주세요\n"
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "text",
          "text": "1. 무슨 상태인지  \n눈 안에 있는 투명한 렌즈가 뿌옇게 변해서, 시야가 흐려지는 상태입니다.\n\n2. 왜 문제가 되는지  \n이 렌즈가 뿌옇게 되면, 밖에서 들어오는 빛이 제대로 통과하지 못해 사물이 흐릿하게 보이거나, 밝은 곳에서 눈이 부시게 느껴질 수 있습니다. 시간이 지나면 시력이 점점 더 나빠져서 일상생활에 불편을 줄 수 있습니다.\n\n3. 당장 주의할 점  \n- 밝은 곳에서는 선글라스를 착용하거나 눈을 보호해 주세요.  \n- 시야가 갑자기 더 흐려지거나 불편함이 심해지면 병원에 다시 오세요.  \n- 눈을 비비지 말고, 정기적으로 눈 상태를 확인하세요."
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "혈액암에 대해 설명해주세요\n"
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "text",
          "text": "1. 무슨 상태인지  \n피 속에 있는 세포들이 제대로 자라지 않거나 이상하게 변해서, 몸에 여러 가지 문제가 생기는 상태입니다.\n\n2. 왜 문제가 되는지  \n피 속의 세포들은 우리 몸을 지키고, 산소를 옮기고, 피가 멈추게 하는 역할을 합니다. 이 세포들이 제대로 일을 못 하면, 쉽게 피가 나거나 멍이 들고, 감기에 자주 걸리거나, 몸이 많이 피곤해질 수 있습니다. 치료가 꼭 필요하며, 치료를 받지 않으면 건강에 큰 문제가 생길 수 있습니다.\n\n3. 당장 주의할 점  \n- 몸에 멍이 잘 들거나, 코피가 자주 나면 바로 알려주세요.  \n- 열이 나거나 감기 증상이 있으면 병원에 꼭 말씀해 주세요.  \n- 무리하지 말고, 몸 상태를 잘 살펴보세요."
        }
      ]
    }
  ],
  response_format={
    "type": "text"
  },
  temperature=0.5,
  max_completion_tokens=2048,
  top_p=1,
  frequency_penalty=0,
  presence_penalty=0,
  store=False
)

In [8]:
# 스트림 적용된 채팅
messages = [
    {'role': 'system', 'content': '당신은 병원에서 환자와 보호자에게 진료 내용을 설명하는 전문 간호사입니다.'},
]

print('종료하려면, exit를 입력하세요.\n')

while True:
  user_input = input('User: ')

  # 종료조건 검사
  if user_input.lower().strip() == 'exit':
    print('대화를 종료합니다.')
    break

  # 사용자입력 로그 및 messages에 추가
  # print(f'User: {user_input}')
  messages.append({'role': 'user', 'content': user_input})
  response = client.chat.completions.create(
      model = 'gpt-4.1-mini',
      messages = messages,
      temperature=1,
      top_p=1,
      max_completion_tokens=4096,
      stream=True
  )

  # 스트림 출력처리
  print(f'Assistant: ', end='')
  assistant_message = ''
  for chunk in response:
    content = chunk.choices[0].delta.content
    if content is not None:
      print(content, end='')
      assistant_message += content
  print('\n')

  messages.append({'role': 'assistant', 'content': assistant_message})


종료하려면, exit를 입력하세요.

User: exit
대화를 종료합니다.


## Step 3: Streamlit 앱 구현

아래 코드를 실행하여 Streamlit 앱을 실행합니다.


In [9]:
# Streamlit 앱 실행
# 주의: Colab에서는 Streamlit을 직접 실행하기 어려우므로,
# 로컬 환경에서 실행하거나 ngrok 등을 사용해야 합니다.

# 로컬에서 실행하려면:
# !streamlit run nurseapp.py

# 또는 Colab에서 실행하려면:
import subprocess
import sys

# Streamlit 앱을 백그라운드로 실행
process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "nurseapp.py", "--server.port=8501", "--server.address=0.0.0.0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("Streamlit 앱이 실행되었습니다!")
print("아래 링크를 클릭하거나 터미널에서 확인하세요.")


Streamlit 앱이 실행되었습니다!
아래 링크를 클릭하거나 터미널에서 확인하세요.


In [10]:
# pip install streamlit